# TARNIB Transformer Model

## Imports

In [11]:
# Data handling and preprocessing
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Transformer traininng
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Model evaluation
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

## Load Data

In [12]:
train_df = pd.read_csv("../data/train.csv")
test_df  = pd.read_csv("../data/test.csv")

print(train_df.shape, test_df.shape)
train_df.head()

(9388, 81) (2430, 81)


,subject_id,hadm_id,hospital_expire_flag,gender,age,LoS,ER_LoS,admission_no,days_since_discharge,days_until_death,...,ccs_emb_21,ccs_emb_22,ccs_emb_23,ccs_emb_24,ccs_emb_25,ccs_emb_26,ccs_emb_27,ccs_emb_28,ccs_emb_29,ccs_emb_30
0,10000690,26504700,0,1,86,4.538889,0.205556,1,NaN,575.884722,...,-0.027652,-0.013480,0.015762,-0.035427,-0.006714,0.001981,0.031827,0.017102,-0.015162,0.023146
1,10000690,23280645,0,1,86,7.751389,0.210417,2,71.170833,500.175000,...,-0.003314,-0.001401,-0.034529,0.016691,-0.017230,0.000959,0.014027,0.013297,-0.007667,-0.001504
2,10000690,25860671,0,1,86,9.821528,0.330556,3,39.175000,453.248611,...,-0.003084,-0.001364,-0.001827,-0.000033,-0.009749,-0.007124,0.024730,0.027403,0.007163,0.014003
3,10000690,26146595,0,1,88,1.677778,0.403472,4,442.413194,1.013889,...,0.006004,-0.001361,-0.028550,-0.003648,-0.033128,-0.001682,0.002926,0.007493,-0.011065,0.013586
4,10001919,29897682,0,0,59,1.574306,NaN,1,NaN,244.000000,...,-0.008147,-0.008002,0.027353,0.016642,-0.028913,0.006824,0.018412,0.002947,-0.018814,-0.017418


## Data Preparation


### Filtering Features

In [13]:
## Select features
label = "hospital_expire_flag"

remove_features = [
    "hospital_expire_flag",
    "days_until_death",
    "hadm_id",
    "subject_id",
    "days_since_discharge",
    "ER_LoS"
]

feature_cols = [c for c in train_df.columns if c not in remove_features]

X_train = train_df[feature_cols].values
y_train = train_df[label].values

X_test = test_df[feature_cols].values
y_test = test_df[label].values

### Normalisation

In [14]:
# Numeric feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Conversion to Dataset format. Quantisation.

In [15]:
class TabDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_ds = TabDataset(X_train, y_train)
test_ds  = TabDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

## Developing and Training

### Model Architecture

In [16]:
class TabTransformer(nn.Module):
    def __init__(self, num_features, d_model=64, nhead=4, num_layers=3, dropout=0.3):
        super().__init__()

        self.feature_embed = nn.Linear(1, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=128,
            dropout=dropout,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.mlp = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = x.unsqueeze(-1)          # (batch, features, 1)
        x = self.feature_embed(x)    # (batch, features, d_model)
        x = self.transformer(x)      # (batch, features, d_model)
        x = x.mean(dim=1)            # pool over features
        x = self.mlp(x).squeeze(1)
        return x

### Initialising Model

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TabTransformer(num_features=X_train.shape[1]).to(device)

pos_weight = torch.tensor([(len(y_train) - y_train.sum()) / y_train.sum()]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

### Training setup

In [18]:
def train_epoch(model, loader):
    model.train()
    total_loss = 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


### Evaluation Function

In [19]:
def eval_model(model, loader):
    model.eval()
    preds, trues = [], []

    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            logits = model(X)
            probs = torch.sigmoid(logits)

            preds.extend(probs.cpu().numpy())
            trues.extend(y.numpy())

    preds = np.array(preds)
    trues = np.array(trues)

    return {
        "AUROC": roc_auc_score(trues, preds),
        "AUPRC": average_precision_score(trues, preds),
        "Brier": brier_score_loss(trues, preds)
    }


### Training Loop

In [20]:
for epoch in range(1, 31):
    loss = train_epoch(model, train_loader)
    metrics = eval_model(model, test_loader)

    print(
        f"Epoch {epoch:02d} | "
        f"Loss {loss:.4f} | "
        f"AUROC {metrics['AUROC']:.4f} | "
        f"AUPRC {metrics['AUPRC']:.4f}"
    )


Epoch 01 | Loss 1.3751 | AUROC 0.4746 | AUPRC 0.0243
Epoch 02 | Loss 1.3610 | AUROC 0.5821 | AUPRC 0.0315
Epoch 03 | Loss 1.3531 | AUROC 0.5879 | AUPRC 0.0321
Epoch 04 | Loss 1.3557 | AUROC 0.5872 | AUPRC 0.0323
Epoch 05 | Loss 1.3548 | AUROC 0.5866 | AUPRC 0.0321
Epoch 06 | Loss 1.3547 | AUROC 0.5000 | AUPRC 0.0226
Epoch 07 | Loss 1.3544 | AUROC 0.5000 | AUPRC 0.0226
Epoch 08 | Loss 1.3555 | AUROC 0.5000 | AUPRC 0.0226
Epoch 09 | Loss 1.3555 | AUROC 0.5000 | AUPRC 0.0226
Epoch 10 | Loss 1.3541 | AUROC 0.5000 | AUPRC 0.0226
Epoch 11 | Loss 1.3529 | AUROC 0.5000 | AUPRC 0.0226
Epoch 12 | Loss 1.3555 | AUROC 0.5000 | AUPRC 0.0226
Epoch 13 | Loss 1.3529 | AUROC 0.5000 | AUPRC 0.0226
Epoch 14 | Loss 1.3542 | AUROC 0.5000 | AUPRC 0.0226
Epoch 15 | Loss 1.3528 | AUROC 0.5000 | AUPRC 0.0226
Epoch 16 | Loss 1.3528 | AUROC 0.5000 | AUPRC 0.0226
Epoch 17 | Loss 1.3528 | AUROC 0.5000 | AUPRC 0.0226
Epoch 18 | Loss 1.3554 | AUROC 0.5000 | AUPRC 0.0226
Epoch 19 | Loss 1.3541 | AUROC 0.5000 | AUPRC 

## Final Metrics

In [21]:
metrics = eval_model(model, test_loader)
metrics

{'AUROC': 0.5, 'AUPRC': 0.02263374485596708, 'Brier': 0.2490721195936203}